# 03 — Publish results to the databases (optional)

The dashboard runs without any database (`DB_MODE=files`, the default), reading
`results/` directly. This notebook loads the same files into PostgreSQL/PostGIS
and MongoDB for `DB_MODE=local`, and optionally into Supabase and MongoDB Atlas
for `DB_MODE=cloud`. It copies; it never computes. Every table is rebuilt from
scratch, so re-running is safe.

| Store | Holds |
|---|---|
| PostGIS `grid_cells` | one polygon per cell, keyed by `cell_id`, with its carbon values and their sources |
| PostgreSQL `district_summary` | one row per pipeline run: headline figures and intervals, plus the full JSON |
| MongoDB `district_summary`, `model_metrics`, `ndvi_monthly` | the result documents as written by the pipeline |
| MongoDB `cells` | the per-cell table without geometry, for the explorer |

Credentials come from `.env` in the project root (see `.env.example`).

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import config as C

load_dotenv(ROOT / ".env")
summary = json.loads((C.RESULTS / "district_summary.json").read_text())
metrics = json.loads((C.RESULTS / "model_metrics.json").read_text())
ndvi = json.loads((C.RESULTS / "ndvi_monthly.json").read_text())
cells = pd.read_csv(C.RESULTS / "carbon_cells.csv")
C.assert_unique(cells)
print(f"run {summary['run_date']} ({summary['mode']}): {len(cells):,} cells")

## Functions

In [ ]:
CELL_FIELDS = ["cell_id", "agri_class", "Ag/NonAg_m", "DEM_mean", "Slope_mean", "soil_status", "w_soil",
               "bd_gcm3_rec", "soc_gkg", "soc_stock_tc_ha", "soc_source", "npp_flux_tc_ha_yr", "npp_source"]
PG_COLS = ["cell_id", "agri_class", "agnonag_m", "dem_mean", "slope_mean", "soil_status", "w_soil",
           "bd_gcm3_rec", "soc_gkg", "soc_stock_tc_ha", "soc_source", "npp_flux_tc_ha_yr", "npp_source"]


def clean(v):
    return None if pd.isna(v) else v


def publish_mongo(db):
    for name, doc in [("district_summary", summary), ("model_metrics", metrics)]:
        db[name].drop()
        db[name].insert_one(dict(doc))
    db["ndvi_monthly"].drop()
    db["ndvi_monthly"].insert_many([dict(d) for d in ndvi])
    db["cells"].drop()
    recs = [{k: clean(v) for k, v in r.items()} for r in cells[CELL_FIELDS].to_dict(orient="records")]
    for i in range(0, len(recs), 10_000):
        db["cells"].insert_many(recs[i:i + 10_000])
    db["cells"].create_index("cell_id", unique=True)
    return {n: db[n].count_documents({}) for n in ["district_summary", "model_metrics", "ndvi_monthly", "cells"]}


def publish_postgres(conn):
    from psycopg2.extras import Json, execute_values

    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS postgis")
    cur.execute("DROP TABLE IF EXISTS grid_cells")
    cur.execute("""
        CREATE TABLE grid_cells (
            cell_id            TEXT PRIMARY KEY,
            geom               GEOMETRY(Polygon, 4326) NOT NULL,
            agri_class         SMALLINT,
            agnonag_m          REAL,
            dem_mean           REAL,
            slope_mean         REAL,
            soil_status        TEXT,
            w_soil             REAL,
            bd_gcm3_rec        REAL,
            soc_gkg            REAL,
            soc_stock_tc_ha    REAL,
            soc_source         TEXT,
            npp_flux_tc_ha_yr  REAL,
            npp_source         TEXT
        )""")
    rows = [tuple(clean(r[c]) for c in CELL_FIELDS) + (r["WKT"],) for r in cells.to_dict(orient="records")]
    execute_values(cur, f"""
        INSERT INTO grid_cells ({", ".join(PG_COLS)}, geom)
        VALUES %s""", rows,
        template="(" + ", ".join(["%s"] * len(PG_COLS)) + ", ST_GeometryN(ST_GeomFromText(%s, 4326), 1))",
        page_size=2000)
    cur.execute("CREATE INDEX grid_cells_geom_idx ON grid_cells USING GIST (geom)")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS district_summary (
            id                 SERIAL PRIMARY KEY,
            run_date           DATE NOT NULL,
            mode               TEXT NOT NULL,
            soil_stock_mtc     DOUBLE PRECISION,
            soil_stock_ci_lo   DOUBLE PRECISION,
            soil_stock_ci_hi   DOUBLE PRECISION,
            npp_flux_mtc_yr    DOUBLE PRECISION,
            npp_flux_ci_lo     DOUBLE PRECISION,
            npp_flux_ci_hi     DOUBLE PRECISION,
            payload            JSONB NOT NULL
        )""")
    s, f = summary["soil"]["stock_mtc"], summary["npp"]["flux_mtc_per_year"]
    ci = lambda e, i: e["ci95"][i] if e.get("ci95") else None
    cur.execute("DELETE FROM district_summary WHERE run_date = %s AND mode = %s", (summary["run_date"], summary["mode"]))
    cur.execute("""INSERT INTO district_summary (run_date, mode, soil_stock_mtc, soil_stock_ci_lo, soil_stock_ci_hi,
                   npp_flux_mtc_yr, npp_flux_ci_lo, npp_flux_ci_hi, payload) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
                (summary["run_date"], summary["mode"], s["estimate"], ci(s, 0), ci(s, 1),
                 f["estimate"], ci(f, 0), ci(f, 1), Json(summary)))
    conn.commit()
    cur.execute("SELECT count(*), count(DISTINCT cell_id) FROM grid_cells")
    n, n_unique = cur.fetchone()
    assert n == n_unique == len(cells)
    return {"grid_cells": n}

## Local databases (`DB_MODE=local`)

In [ ]:
from pymongo import MongoClient
import psycopg2

mongo = MongoClient(os.getenv("MONGO_URI", "mongodb://localhost:27017/"), serverSelectionTimeoutMS=5000)
mongo.admin.command("ping")
print("MongoDB:", publish_mongo(mongo[os.getenv("MONGO_DB", "carbon_stock_ludhiana")]))

if not os.getenv("PG_PASSWORD"):
    raise ValueError("PG_PASSWORD missing from .env")
conn = psycopg2.connect(host=os.getenv("PG_HOST", "localhost"), port=int(os.getenv("PG_PORT", "5432")),
                        dbname=os.getenv("PG_DB", "postgres"), user=os.getenv("PG_USER", "postgres"),
                        password=os.getenv("PG_PASSWORD"))
try:
    print("PostgreSQL:", publish_postgres(conn))
finally:
    conn.close()
    mongo.close()

## Cloud (`DB_MODE=cloud`, optional) — skipped unless the credentials are in `.env`

In [ ]:
if os.getenv("ATLAS_URI"):
    atlas = MongoClient(os.getenv("ATLAS_URI"), serverSelectionTimeoutMS=10_000)
    print("Atlas:", publish_mongo(atlas[os.getenv("MONGO_DB", "carbon_stock_ludhiana")]))
    atlas.close()
else:
    print("ATLAS_URI not set; skipped")

if os.getenv("SUPABASE_PASSWORD"):
    sconn = psycopg2.connect(host=os.getenv("SUPABASE_HOST"), port=int(os.getenv("SUPABASE_PORT", "5432")),
                             dbname=os.getenv("SUPABASE_DB", "postgres"), user=os.getenv("SUPABASE_USER"),
                             password=os.getenv("SUPABASE_PASSWORD"), sslmode="require")
    try:
        print("Supabase:", publish_postgres(sconn))
    finally:
        sconn.close()
else:
    print("SUPABASE_PASSWORD not set; skipped")